In [1]:
import lightgbm
import mlflow
import os
from dotenv import load_dotenv
import sys
import pandas as pd
import numpy as np
import json

In [2]:
pd.set_option("display.max_columns", 100)
load_dotenv()
data_path = os.getenv("DATA_PATH")
src_path = os.getenv("SRC_PATH")
sys.path.append(src_path)
json_path = os.path.join(data_path, "processed/split_info.json")
with open(json_path) as f:
    json_info = json.load(f)
train_end = json_info.get("train_end")
val_end = json_info.get("validation_end")
from about_data.data_load import load_df
from about_data.split import temporal_split
from features.engineering import create_d_features, create_advanced_time_features, add_distance_features, add_interaction_features
from model.preprocessor_pipe_evalueate import create_pipeline, evaluate_model, get_preprocessor

full_df = load_df(data_path)
train, val, test = temporal_split(full_df, train_end, val_end)

In [3]:
train_base = create_d_features(train)
val_base = create_d_features(val)
map_dfs = {"train": train, "val": val}
map_functions = {"d":create_d_features, "time":create_advanced_time_features, "distance":add_distance_features, "interaction":add_interaction_features}
train_features_dfs = {}
val_features_dfs = {}

train_features_dfs = {"d": train_base}
val_features_dfs = {"d": val_base}

for name, func in map_functions.items():
    train_features_dfs[name] = func(train_base)
    val_features_dfs[name] = func(val_base)

In [4]:
len(train_features_dfs)

4